# 21 · Opto after the distribution shift (Hard-A / Hard-B)

**What this shows.** The post-shift prediction — PPC necessary when the statistical model has to update —
needs animals that are *adapting* when the laser is on. The opto cohort alternated Hard-A and Hard-B daily.
This notebook shows (1) the same contrasts as notebook 20 on the Hard phases and (2) the per-session
trajectory of the criterion against the normative value, which is the test of whether the daily
alternation produced adaptation at all. It did not measurably, which is the reason the next cohort uses
blocks (notebook 11).

**What produced it.** The same `opto_contrasts` run as notebook 20 (`sc-reports opto --all` covers all
three distributions). Besides `contrasts.csv`, every per-animal folder has `trajectory.csv` (one row per
session in order: `pse`, `normative_pse`, `baseline_pse`, `prev_distribution`, per-session stats) and
`trajectory_curves.csv` (PSE against trials within each session, from `behaviour/adaptation.compute_trajectory`).

In [ ]:
from nb_setup import *
banner()
run = open_run('opto_contrasts', OPTO_COHORT)
C = load_contrasts(run)
rows, tests = load_group(run)
traj = load_trajectory(run)
genotype = C.drop_duplicates('animal').set_index('animal')['genotype']
hard = C[C.distribution.isin(['Hard-A', 'Hard-B']) & (C.design == 'ppc') & (C.toi == 'opto')]
print('Hard distributions present:', sorted(hard.distribution.unique()), '| trajectory rows:', len(traj))

## 1 · One animal: the per-session trajectory from the library

`compute_trajectory(animal, distributions)` returns one row per session: the PSE fitted on that session,
the normative PSE for its distribution, the previous distribution, and the within-session PSE curve.

In [ ]:
from sound_categorisation.behaviour.adaptation import compute_trajectory

exp = load_experiment()
aid = first_animal(hard) if len(hard) else sorted(C.animal.unique())[0]
tr = compute_trajectory(exp.animals[aid], ('Hard-A', 'Hard-B'))
tab = tr.sessions if hasattr(tr, 'sessions') else tr
display(tab[['session_idx', 'distribution', 'prev_distribution', 'n_trials', 'pse', 'normative_pse']].head(12).round(3)
        if len(tab) else 'no Hard sessions for this animal')

## 2 · Did the criterion follow the distribution?

PSE per session for every animal, with the normative PSE of that session's distribution as a thin line.
Under adaptation the points should sit on the alternating normative values; a flat trace means the
daily alternation was too short for the animal to re-estimate anything.

In [ ]:
if len(traj):
    th = traj[traj.distribution.isin(['Hard-A', 'Hard-B'])].sort_values(['animal', 'session_idx'])
    animals = sorted(th.animal.unique())
    fig, axes = plt.subplots(len(animals), 1, figsize=(10, 1.6 * len(animals)), sharex=True, squeeze=False)
    for ax, a in zip(axes[:, 0], animals):
        s = th[th.animal == a]
        ax.plot(s['session_idx'], s['normative_pse'], color='0.75', lw=1, drawstyle='steps-mid', label='normative')
        for d, ss in s.groupby('distribution'):
            ax.scatter(ss['session_idx'], ss['pse'], s=18, label=d)
        ax.axhline(0, color='0.9', lw=0.5)
        ax.set_ylabel(f'{a}\n({genotype.get(a, "?")})', rotation=0, ha='right', va='center', fontsize=8)
    axes[0, 0].legend(fontsize=7, ncol=3); axes[-1, 0].set_xlabel('session')
    plt.tight_layout()
else:
    print('no trajectory tables in this run')

## 3 · Convergence within a session

`trajectory_curves.csv`: PSE against trials inside each Hard session, as `convergence` toward the
normative value. If animals adapted within a day the curves would rise toward 1 over the session.

In [ ]:
import glob
files = sorted(glob.glob(str(run / 'Hard-*' / 'ppc_opto' / '*' / 'trajectory_curves.csv')))
if files:
    tc = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for d, s in tc.groupby('distribution'):
        m = s.groupby('trial')['convergence'].mean()
        ax.plot(m.index, m.values, lw=2, label=f'{d} (n={s.groupby(["animal","session_idx"]).ngroups} sessions)')
    ax.axhline(0, color='0.6', lw=0.6); ax.axhline(1, color='0.6', lw=0.6, ls='--')
    ax.set(xlabel='trial in session', ylabel='convergence toward normative PSE', ylim=(-0.5, 1.5))
    ax.legend(fontsize=8)
else:
    print('no trajectory_curves.csv (run was --fast); rerun sc-reports opto --distribution Hard-A without --fast')

## 4 · The opto contrasts on the Hard phases, per animal and across genotypes

Same reading as notebook 20. With no adaptation to disrupt, the expectation is the expert-phase picture
again, and that is what to check here.

In [ ]:
STATS = ['mu', 'sigma', 'accuracy', 'recency']
def forest(df, stat, ax, kind='within', unit='trials', title=None):
    d = df[(df.kind == kind) & (df.unit == unit) & (df.stat == stat)].copy()
    d['g'] = d.animal.map(genotype)
    d = d.sort_values(['g', 'animal'], ascending=[False, True]).reset_index(drop=True)
    for i, r in d.iterrows():
        c = GENOTYPE_COLOUR.get(r['g'], '0.5')
        ax.errorbar(r['diff'], i, xerr=[[r['diff'] - r['ci_lo']], [r['ci_hi'] - r['diff']]], fmt='o', color=c, ms=4)
    ax.axvline(0, color='0.6', lw=0.8)
    ax.set(yticks=range(len(d)), yticklabels=d.animal, title=title or f'{kind}: {stat}')

for dist in sorted(hard.distribution.unique()):
    h = hard[hard.distribution == dist]
    fig, axes = plt.subplots(1, len(STATS), figsize=(3.2 * len(STATS), 0.35 * h.animal.nunique() + 1.5), sharey=True)
    for ax, st in zip(axes, STATS):
        forest(h, st, ax, title=f'{dist} within: {st}')
    plt.tight_layout()

In [ ]:
t = tests[tests.distribution.isin(['Hard-A', 'Hard-B']) & (tests.design == 'ppc') & (tests.toi == 'opto') & (tests.kind == 'within')]
t[t.stat.isin(STATS)][['distribution', 'stat', 'n_a', 'n_b', 'median_a', 'median_b', 'p', 'min_p']].sort_values(['distribution', 'stat']).round(3)